# Step 3 - Make the baseline corpus (E0)

We take real training questions, let the untrained model reason through each
one, check if it got the answer right, and score the reasoning with the
argument pipeline. Everything is saved to one file.

This is our raw material. The correlation study reads it to ask: do the
argument scores line up with getting the answer right? It is also a finding on
its own: how does the model reason before any training.

No training here, just generation and scoring. Run cells top to bottom.

## Setup: get the repo (force latest code)

In [ ]:
import os, sys, subprocess

REPO = "https://github.com/ookino/rlvr-argument-mining.git"
NAME = "rlvr-argument-mining"

if os.path.basename(os.getcwd()) != NAME:
    if not os.path.isdir(NAME):
        subprocess.run(["git", "clone", REPO], check=True)
    os.chdir(NAME)
subprocess.run(["git", "fetch", "--quiet"], check=False)
subprocess.run(["git", "reset", "--hard", "origin/main"], check=False)

sys.path.insert(0, os.getcwd())
print("repo ready, at", os.getcwd())

## Install what we need

Generation needs transformers and bitsandbytes. Scoring needs networkx. These are usually already on Colab.

In [ ]:
!pip install -q transformers bitsandbytes accelerate datasets networkx pyyaml

## Check the GPU is on

Should print True. If it says False, use Runtime > Change runtime type > GPU (L4), then run from the top.

In [ ]:
import torch
print("gpu:", torch.cuda.is_available())

## Save to Google Drive so nothing is lost

Colab's own disk is temporary: when Colab gives you a new machine, anything
saved there is gone. So we mount your Google Drive and save the corpus there
instead. Drive is permanent, so even a fresh machine can pick up where a
stopped run left off.

Running this pops up a permission window. Say yes. It creates a folder in your
Drive for this project.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
OUT = "/content/drive/MyDrive/rlvr-argument-mining/E0/corpus.jsonl"
os.makedirs(os.path.dirname(OUT), exist_ok=True)
print("corpus will be saved to:", OUT)

## Small test first: 5 questions

Before the full run, do 5 questions to check the whole thing works: the model
generates, the answer is read, the argument scores compute, and it saves. This
takes a couple of minutes (it downloads the model the first time).

In [ ]:
from e0_corpus import generate

generate(n_questions=5, out=OUT)

## IMPORTANT: wipe the old corpus first

Run this ONCE before the full run below. It deletes the old corpus file so the
new run starts fresh. If you skip it, the new traces get appended to the old
ones and the file ends up mixing old and new data.

In [ ]:
from e0_corpus import reset_corpus
reset_corpus(OUT)

## The full run: 300 questions

Shuffled across all the training families, roughly 35 to 40 per family. Slow because the model writes a full trace for each one, one at a time. Expect roughly 45 to 75 minutes.

Safe to stop and re-run: the saver skips questions already done, so if the session dies you just run this cell again and it carries on.

In [ ]:
from e0_corpus import generate

generate(n_questions=300, out=OUT)

## Look at what we made

Prints how many the model got right, how often answer-reading failed, and the
average argument scores. This is the first real look at how the untrained model
reasons.

In [ ]:
from e0_corpus import summarise

summarise(OUT)

## What to do next

- If the numbers look sensible: tell Claude and we build the correlation study
  (does argument structure predict correct answers?).
- If something looks off (all wrong, all zeros, lots of extraction failures):
  paste the output back.

## Check the file is really saved

Confirms the corpus file exists in your Drive and shows how many traces are in
it. This is how you know saving worked.

In [ ]:
import os
from utils import read_jsonl
print("exists:", os.path.exists(OUT))
print("size:", os.path.getsize(OUT), "bytes")
print("traces saved:", sum(1 for _ in read_jsonl(OUT)))